In [11]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE_URL = "https://www.federalreserve.gov"
MAIN_URL = f"{BASE_URL}/newsevents/pressreleases.htm"

response = session.get(MAIN_URL, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

archive_urls = {}

for a in soup.select("a[href]"):
    label = a.get_text(" ", strip=True)
    href = urljoin(MAIN_URL, a["href"])

    # ข้อความลิงก์ต้องเป็น "2015" หรือ "2015 FOMC"
    match = re.fullmatch(r"(20\d{2})(?:\s+FOMC)?", label)
    if not match:
        continue

    year = int(match.group(1))
    if not 2006 <= year <= 2026:
        continue

    parsed = urlparse(href)
    if parsed.scheme != "https":
        continue
    if parsed.netloc != "www.federalreserve.gov":
        continue

    if "/newsevents/pressreleases/" not in parsed.path:
        continue

    archive_type = "fomc" if label.endswith("FOMC") else "general"
    archive_urls[(year, archive_type)] = href

df_archives = pd.DataFrame([
    {
        "year": year,
        "archive_type": kind,
        "archive_url": url,
    }
    for (year, kind), url in sorted(archive_urls.items())
])

display(df_archives.to_string(index=False))

found_years = set(df_archives["year"]) if not df_archives.empty else set()
print("Archive pages:", len(df_archives))
print("Missing years:", sorted(set(range(2006, 2027)) - found_years))

' year archive_type                                                                 archive_url\n 2006      general         https://www.federalreserve.gov/newsevents/pressreleases/2006all.htm\n 2007      general         https://www.federalreserve.gov/newsevents/pressreleases/2007all.htm\n 2008      general         https://www.federalreserve.gov/newsevents/pressreleases/2008all.htm\n 2009      general         https://www.federalreserve.gov/newsevents/pressreleases/2009all.htm\n 2010      general         https://www.federalreserve.gov/newsevents/pressreleases/2010all.htm\n 2011      general         https://www.federalreserve.gov/newsevents/pressreleases/2011all.htm\n 2012      general         https://www.federalreserve.gov/newsevents/pressreleases/2012all.htm\n 2013      general         https://www.federalreserve.gov/newsevents/pressreleases/2013all.htm\n 2014      general         https://www.federalreserve.gov/newsevents/pressreleases/2014all.htm\n 2015      general         https://www.

Archive pages: 28
Missing years: []


In [17]:

import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def extract_release_links(archive_url):
    response = session.get(archive_url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for a in soup.select("a[href]"):
        title = a.get_text(" ", strip=True)
        url = urljoin(archive_url, a["href"])
        parsed = urlparse(url)
        path = parsed.path
        filename = path.rsplit("/", 1)[-1]

        # ต้องเป็น HTTPS และโดเมน Fed ทางการ
        if parsed.scheme != "https":
            continue
        if parsed.netloc != "www.federalreserve.gov":
            continue

        # เอาเฉพาะหน้าประกาศ ไม่เอา archive หรือ navigation
        if "/newsevents/pressreleases/" not in path:
            continue

        if re.fullmatch(r"20\d{2}all\.htm", filename):
            continue

        if re.fullmatch(r"20\d{2}-press(-fomc)?\.htm", filename):
            continue

        if not title:
            continue

        if title.lower() in {
            "skip to main content",
            "back to top",
        }:
            continue

        rows.append({
            "archive_url": archive_url,
            "title": title,
            "url": url,
        })

    # ตัด URL ซ้ำภายใน archive
    return list({row["url"]: row for row in rows}.values())



all_release_links = []
archive_audit = []

for archive in df_archives.to_dict("records"):
    try:
        links = extract_release_links(archive["archive_url"])

        archive_audit.append({
            **archive,
            "release_links": len(links),
            "status": "OK"
        })

        for row in links:
            all_release_links.append({
                **row,
                "archive_year": archive["year"],
                "archive_type": archive["archive_type"]
            })

    except Exception as exc:
        archive_audit.append({
            **archive,
            "release_links": 0,
            "status": f"ERROR: {exc}"
        })

    time.sleep(0.3)

df_audit = pd.DataFrame(archive_audit)
df_index_v2 = pd.DataFrame(all_release_links)

if not df_index_v2.empty:
    df_index_v2 = df_index_v2.drop_duplicates(subset=["url"])

display(df_audit)
print("Unique release URLs:", len(df_index_v2))
print("Years found:", sorted(df_index_v2["archive_year"].unique()))

,year,archive_type,archive_url,release_links,status
0,2006,general,https://www.federalreserve.gov/newsevents/pres...,197,OK
1,2007,general,https://www.federalreserve.gov/newsevents/pres...,191,OK
2,2008,general,https://www.federalreserve.gov/newsevents/pres...,337,OK
3,2009,general,https://www.federalreserve.gov/newsevents/pres...,400,OK
4,2010,general,https://www.federalreserve.gov/newsevents/pres...,400,OK
5,2011,general,https://www.federalreserve.gov/newsevents/pres...,236,OK
6,2012,general,https://www.federalreserve.gov/newsevents/pres...,209,OK
7,2013,general,https://www.federalreserve.gov/newsevents/pres...,216,OK
8,2014,general,https://www.federalreserve.gov/newsevents/pres...,253,OK
9,2015,general,https://www.federalreserve.gov/newsevents/pres...,230,OK


Unique release URLs: 4629
Years found: [np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [18]:

import pandas as pd
import time
from tqdm.auto import tqdm

all_release_links = []
archive_audit = []

for archive in tqdm(df_archives.to_dict("records")):
    try:
        links = extract_release_links(archive["archive_url"])

        archive_audit.append({
            **archive,
            "release_links": len(links),
            "status": "OK",
        })

        for row in links:
            all_release_links.append({
                **row,
                "archive_year": archive["year"],
                "archive_type": archive["archive_type"],
            })

    except Exception as exc:
        archive_audit.append({
            **archive,
            "release_links": 0,
            "status": f"ERROR: {exc}",
        })

    time.sleep(0.3)

df_index_v2 = pd.DataFrame(all_release_links)

if not df_index_v2.empty:
    df_index_v2 = df_index_v2.drop_duplicates(
        subset=["url"]
    ).reset_index(drop=True)

df_audit = pd.DataFrame(archive_audit)

print("Total rows:", len(df_index_v2))
print("Unique URLs:", df_index_v2["url"].nunique())
print(
    "Navigation titles:",
    df_index_v2["title"].str.lower().isin([
        "skip to main content",
        "back to top",
    ]).sum()
)

display(df_audit)


  0%|          | 0/28 [00:00<?, ?it/s]

Total rows: 4629
Unique URLs: 4629
Navigation titles: 0


,year,archive_type,archive_url,release_links,status
0,2006,general,https://www.federalreserve.gov/newsevents/pres...,197,OK
1,2007,general,https://www.federalreserve.gov/newsevents/pres...,191,OK
2,2008,general,https://www.federalreserve.gov/newsevents/pres...,337,OK
3,2009,general,https://www.federalreserve.gov/newsevents/pres...,400,OK
4,2010,general,https://www.federalreserve.gov/newsevents/pres...,400,OK
5,2011,general,https://www.federalreserve.gov/newsevents/pres...,236,OK
6,2012,general,https://www.federalreserve.gov/newsevents/pres...,209,OK
7,2013,general,https://www.federalreserve.gov/newsevents/pres...,216,OK
8,2014,general,https://www.federalreserve.gov/newsevents/pres...,253,OK
9,2015,general,https://www.federalreserve.gov/newsevents/pres...,230,OK


In [19]:
display(
    df_index_v2.groupby("archive_year").agg(
        records=("url", "size"),
        unique_urls=("url", "nunique"),
        first_title=("title", "first"),
    ).reset_index()
)

df_index_v2.to_csv(
    OUTPUT_DIR / "fed_press_index_v3.csv",
    index=False,
    encoding="utf-8-sig",
)

,archive_year,records,unique_urls,first_title
0,2006,197,197,Annual notice of asset-size exemption threshol...
1,2007,191,191,Federal Reserve announces results of auction o...
2,2008,337,337,Six new members named to the Thrift Institutio...
3,2009,400,400,Restructuring of check processing operations i...
4,2010,400,400,Board issues enforcement actions with Baylake ...
5,2011,236,236,Federal Reserve Board issues enforcement actio...
6,2012,209,209,Agencies release annual CRA asset-size thresho...
7,2013,216,216,Agencies reviewing treatment of collateralized...
8,2014,253,253,Federal Reserve Board announces appraisals for...
9,2015,230,230,Federal Reserve Board announces approval of ap...


In [15]:
# ตรวจชื่อที่ดูเหมือนเป็นลิงก์นำทาง
bad_titles = [
    "skip to main content",
    "home",
    "back to top",
]

df_index_v2["title_clean"] = (
    df_index_v2["title"]
    .fillna("")
    .str.strip()
    .str.lower()
)

df_bad = df_index_v2[
    df_index_v2["title_clean"].isin(bad_titles)
]

print("Potential navigation links:", len(df_bad))
display(
    df_bad[
        ["archive_year", "archive_type", "title", "url", "archive_url"]
    ].head(20)
)

Potential navigation links: 20


,archive_year,archive_type,title,url,archive_url
0,2006,general,Skip to main content,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
198,2006,general,Back to Top,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
199,2007,general,Skip to main content,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
391,2007,general,Back to Top,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
392,2008,general,Skip to main content,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
730,2008,general,Back to Top,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
731,2009,general,Skip to main content,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
1132,2009,general,Back to Top,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
1133,2010,general,Skip to main content,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...
1534,2010,general,Back to Top,https://www.federalreserve.gov/newsevents/pres...,https://www.federalreserve.gov/newsevents/pres...


In [16]:
from urllib.parse import urlparse

display(
    df_bad.assign(
        path=df_bad["url"].map(lambda u: urlparse(u).path)
    )[
        ["archive_year", "title", "path", "archive_url"]
    ].head(30)
)

,archive_year,title,path,archive_url
0,2006,Skip to main content,/newsevents/pressreleases/2006all.htm,https://www.federalreserve.gov/newsevents/pres...
198,2006,Back to Top,/newsevents/pressreleases/2006all.htm,https://www.federalreserve.gov/newsevents/pres...
199,2007,Skip to main content,/newsevents/pressreleases/2007all.htm,https://www.federalreserve.gov/newsevents/pres...
391,2007,Back to Top,/newsevents/pressreleases/2007all.htm,https://www.federalreserve.gov/newsevents/pres...
392,2008,Skip to main content,/newsevents/pressreleases/2008all.htm,https://www.federalreserve.gov/newsevents/pres...
730,2008,Back to Top,/newsevents/pressreleases/2008all.htm,https://www.federalreserve.gov/newsevents/pres...
731,2009,Skip to main content,/newsevents/pressreleases/2009all.htm,https://www.federalreserve.gov/newsevents/pres...
1132,2009,Back to Top,/newsevents/pressreleases/2009all.htm,https://www.federalreserve.gov/newsevents/pres...
1133,2010,Skip to main content,/newsevents/pressreleases/2010all.htm,https://www.federalreserve.gov/newsevents/pres...
1534,2010,Back to Top,/newsevents/pressreleases/2010all.htm,https://www.federalreserve.gov/newsevents/pres...


In [20]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse

sample_years = [2006, 2010, 2015, 2020, 2026]
samples = (
    df_index_v2[
        df_index_v2["archive_year"].isin(sample_years)
    ]
    .groupby("archive_year", group_keys=False)
    .head(2)
)

def inspect_article(url):
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.title.get_text(" ", strip=True) if soup.title else None

        for tag in soup.select(
            "script, style, nav, footer, header, form, noscript"
        ):
            tag.decompose()

        main = (
            soup.select_one("#article")
            or soup.select_one("main")
            or soup.select_one(".col-xs-12")
            or soup.body
            or soup
        )

        content = main.get_text("\n", strip=True)

        return {
            "status": r.status_code,
            "final_url": r.url,
            "page_title": title,
            "content_chars": len(content),
            "content_preview": content[:250],
        }

    except Exception as e:
        return {
            "status": "ERROR",
            "final_url": url,
            "page_title": None,
            "content_chars": 0,
            "content_preview": str(e),
        }

test_results = []

for row in samples.to_dict("records"):
    result = inspect_article(row["url"])
    test_results.append({
        "archive_year": row["archive_year"],
        "index_title": row["title"],
        "url": row["url"],
        **result,
    })

display(pd.DataFrame(test_results))

,archive_year,index_title,url,status,final_url,page_title,content_chars,content_preview
0,2006,Annual notice of asset-size exemption threshol...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Annual notice of asset...,1640,"December 29, 2006\nAnnual notice of asset-size..."
1,2006,"Board will be closed on January 2, 2007, in ob...",https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Board will be closed o...,672,"December 28, 2006\nBoard will be closed on Jan..."
2,2010,Board issues enforcement actions with Baylake ...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Board issues enforceme...,611,"December 30, 2010\nBoard issues enforcement ac..."
3,2010,Board issues enforcement action against BankEast,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Board issues enforceme...,343,"December 28, 2010\nBoard issues enforcement ac..."
4,2015,Federal Reserve Board announces approval of ap...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve Board ...,410,"December 31, 2015\nFederal Reserve Board annou..."
5,2015,Federal Reserve Board announces approval of ap...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve Board ...,908,"December 30, 2015\nFederal Reserve Board annou..."
6,2020,Federal Reserve issues FOMC statement,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve issues...,3266,"December 16, 2020\nFederal Reserve issues FOMC..."
7,2020,Federal Reserve Board and Federal Open Market ...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve Board ...,442,"December 16, 2020\nFederal Reserve Board and F..."
8,2026,Federal Reserve issues FOMC statement,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve issues...,1057,"September 16, 2026\nFederal Reserve issues FOM..."
9,2026,Federal Reserve Board and Federal Open Market ...,https://www.federalreserve.gov/newsevents/pres...,200,https://www.federalreserve.gov/newsevents/pres...,Federal Reserve Board - Federal Reserve Board ...,472,"September 16, 2026\nFederal Reserve Board and ..."


In [21]:
import json
import time
import re
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from tqdm.auto import tqdm
from pathlib import Path

DATA_FILE = OUTPUT_DIR / "fed_press_releases.jsonl"
ERROR_FILE = OUTPUT_DIR / "fed_download_errors.jsonl"

def get_published_at(soup):
    # 1. Metadata มักให้วันที่ที่เป็นมาตรฐานที่สุด
    for selector in [
        'meta[property="article:published_time"]',
        'meta[name="DC.date"]',
        'meta[name="date"]',
        'meta[name="dcterms.date"]',
    ]:
        tag = soup.select_one(selector)
        if tag and tag.get("content"):
            value = tag["content"].strip()
            try:
                return datetime.fromisoformat(
                    value.replace("Z", "+00:00")
                ).isoformat()
            except ValueError:
                pass

    # 2. ลองหา Release Date ในข้อความหน้าเว็บ
    text = soup.get_text(" ", strip=True)
    match = re.search(
        r"Release Date:\s*"
        r"([A-Z][a-z]+\s+\d{1,2},\s+\d{4})",
        text,
        flags=re.IGNORECASE,
    )

    if match:
        try:
            return datetime.strptime(
                match.group(1), "%B %d, %Y"
            ).date().isoformat()
        except ValueError:
            pass

    return None


def fetch_article(row):
    response = session.get(row["url"], timeout=30)
    response.raise_for_status()

    if "text/html" not in response.headers.get("Content-Type", ""):
        raise ValueError("Response is not HTML")

    soup = BeautifulSoup(response.text, "html.parser")

    page_title = (
        soup.title.get_text(" ", strip=True)
        if soup.title else None
    )
    published_at = get_published_at(soup)

    for tag in soup.select(
        "script, style, nav, footer, header, form, noscript"
    ):
        tag.decompose()

    main = (
        soup.select_one("#article")
        or soup.select_one("main")
        or soup.select_one(".col-xs-12")
        or soup.body
        or soup
    )

    content = main.get_text("\n", strip=True)

    if len(content) < 200:
        raise ValueError(f"Content too short: {len(content)} chars")

    return {
        **row,
        "source": "Federal Reserve Board",
        "page_title": page_title,
        "published_at": published_at,
        "collected_at": datetime.now(timezone.utc).isoformat(),
        "content": content,
        "fetch_error": None,
    }


# ตรวจสอบ URL ที่ดาวน์โหลดสำเร็จแล้ว
done_urls = set()

if DATA_FILE.exists():
    with DATA_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                if item.get("content") and not item.get("fetch_error"):
                    done_urls.add(item["url"])
            except (json.JSONDecodeError, KeyError):
                continue

errors = []

with DATA_FILE.open("a", encoding="utf-8") as out:
    for row in tqdm(df_index_v2.to_dict("records")):
        if row["url"] in done_urls:
            continue

        record = None

        # Retry สูงสุด 3 ครั้ง
        for attempt in range(3):
            try:
                record = fetch_article(row)
                break
            except Exception as exc:
                error_message = str(exc)
                time.sleep(1.5 * (attempt + 1))

        if record is None:
            errors.append({
                "url": row["url"],
                "archive_year": row["archive_year"],
                "error": error_message,
                "collected_at": datetime.now(timezone.utc).isoformat(),
            })
        else:
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            done_urls.add(row["url"])

        time.sleep(0.4)

if errors:
    with ERROR_FILE.open("a", encoding="utf-8") as f:
        for item in errors:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Successful unique URLs:", len(done_urls))
print("Failed in this run:", len(errors))
print("Data file:", DATA_FILE)

  0%|          | 0/4629 [00:00<?, ?it/s]

Successful unique URLs: 4658
Failed in this run: 0
Data file: dataset\news\fed\fed_press_releases.jsonl


In [22]:
records = []

with DATA_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df_fed = pd.DataFrame(records)

print("Records:", len(df_fed))
print("Unique URLs:", df_fed["url"].nunique())
print("Missing published_at:", df_fed["published_at"].isna().sum())
print("Missing content:", df_fed["content"].isna().sum())

if not df_fed.empty:
    df_fed["published_date"] = pd.to_datetime(
        df_fed["published_at"], errors="coerce", utc=True
    )

    display(
        df_fed.groupby("archive_year").agg(
            records=("url", "size"),
            dated=("published_date", "count"),
            missing_dates=("published_date", lambda s: s.isna().sum()),
            avg_content_chars=("content", lambda s: s.str.len().mean()),
        ).reset_index()
    )

Records: 4658
Unique URLs: 4658
Missing published_at: 4658
Missing content: 0


,archive_year,records,dated,missing_dates,avg_content_chars
0,2006.0,197,0,197,1350.796954
1,2007.0,191,0,191,1615.408377
2,2008.0,337,0,337,1486.884273
3,2009.0,400,0,400,1262.040000
4,2010.0,400,0,400,1014.960000
5,2011.0,236,0,236,1503.563559
6,2012.0,209,0,209,1625.645933
7,2013.0,216,0,216,1635.587963
8,2014.0,253,0,253,1698.474308
9,2015.0,230,0,230,1614.773913
